# S04E03 — Domatowo (Misja Ratunkowa)

Zadanie: odnalezienie partyzanta ukrywającego się w ruinach Domatowa i przeprowadzenie sprawnej akcji ewakuacyjnej.

Wskazówka z sygnału radiowego:
> "Ukryłem się w jednym z najwyższych bloków."

Zasoby:
- maks. 4 transportery
- maks. 8 zwiadowców
- 300 punktów akcji
- mapa 11x11

Koszty:
- zwiadowca: 5 pkt
- transporter (+ 5 za każdego pasażera): 5 pkt bazowo
- ruch zwiadowcy: 7 pkt/pole
- ruch transportera: 1 pkt/pole
- inspekcja pola: 1 pkt

Podejście agentowe: Claude analizuje mapę, planuje trasę, zarządza jednostkami i szuka partyzanta.

In [ ]:
import os, json, requests
from dotenv import load_dotenv
import anthropic

load_dotenv("../.env")
API_KEY = os.getenv("AI_DEVS_API_KEY")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

VERIFY_URL = "https://hub.ag3nts.org/verify"
TASK = "domatowo"

client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

In [ ]:
# Funkcja pomocnicza — wywołanie API Domatowo
def domatowo_api(answer: dict) -> dict:
    payload = {"apikey": API_KEY, "task": TASK, "answer": answer}
    r = requests.post(VERIFY_URL, json=payload)
    result = r.json()
    return result

def show_api(answer: dict) -> dict:
    result = domatowo_api(answer)
    print(f">> {json.dumps(answer)[:100]}")
    print(f"<< {json.dumps(result, ensure_ascii=False)[:400]}")
    return result

# Sprawdź dostępne akcje
show_api({"action": "help"})

In [ ]:
# Pobierz mapę miasta
map_result = show_api({"action": "getMap"})
print("\nMapa Domatowa:")
print(json.dumps(map_result, indent=2, ensure_ascii=False)[:2000])

## Definicja narzędzi agenta

In [ ]:
TOOLS = [
    {
        "name": "domatowo_action",
        "description": (
            "Wykonuje akcję w grze misji ratunkowej Domatowo. "
            "Dostępne akcje: help, getMap, create (type: 'transporter'|'scout', passengers: liczba), "
            "move (unitId, direction: N/S/E/W, steps), inspect (unitId, position), "
            "getLogs, disembark (transporterId), callHelicopter (destination: np. 'F6'). "
            "Każda akcja zużywa punkty — planuj oszczędnie (masz 300 pkt łącznie)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "answer": {
                    "type": "object",
                    "description": "Obiekt akcji (musi zawierać pole 'action' i opcjonalne parametry)"
                }
            },
            "required": ["answer"]
        }
    }
]

action_log = []  # historia wszystkich akcji i wyników

def handle_tool(name: str, input_data: dict) -> str:
    if name == "domatowo_action":
        answer = input_data["answer"]
        result = domatowo_api(answer)
        entry = {"action": answer, "result": result}
        action_log.append(entry)
        print(f"  >> {json.dumps(answer)[:150]}")
        print(f"  << {json.dumps(result, ensure_ascii=False)[:400]}")
        return json.dumps(result, ensure_ascii=False)
    return f"Nieznane narzędzie: {name}"

## Agent — pętla ReAct

In [ ]:
SYSTEM_PROMPT = """Jesteś dowódcą misji ratunkowej w zniszczonym mieście Domatowo.

CEL: Odnaleźć ukrywającego się człowieka i wezwać helikopter ratunkowy.

WSKAZÓWKA: Człowiek ukrył się w "jednym z najwyższych bloków" - szukaj bloków wielopiętrowych.

ZASOBY (PILNUJ BUDŻETU!):
- Masz 300 punktów akcji łącznie
- Transporter (T): kosztuje 5 + (5 × pasażerowie) do stworzenia, porusza się za 1 pkt/pole
- Zwiadowca (S): kosztuje 5 do stworzenia, porusza się za 7 pkt/pole
- Inspekcja pola: 1 pkt

STRATEGIA (użyj tej kolejności):
1. Pobierz pomoc (help) by poznać dokładne API.
2. Pobierz mapę (getMap) i dokładnie przeanalizuj symbole terenu.
3. Zidentyfikuj "najwyższe bloki" na mapie (zazwyczaj oznaczone specjalnym symbolem).
4. Stwórz transporter z 1-2 zwiadowcami i jedź do obszaru bloków.
5. Wysadź zwiadowców i inspektuj pola bloków.
6. Sprawdzaj logi (getLogs) po każdym zestawie inspekcji.
7. Gdy znajdziesz człowieka, NATYCHMIAST wezwij helikopter (callHelicopter).

WAŻNE:
- Transportery jeżdżą tylko po ulicach.
- Zwiadowcy mogą chodzić po każdym terenie.
- Oszczędzaj punkty — używaj transportera do przemieszczania zwiadowców.
- Gdy w odpowiedzi pojawi się flaga (FLG:), wyświetl ją natychmiast."""

messages = [
    {"role": "user", "content": "Zacznij misję ratunkową w Domatowie. Znajdź ukrytego człowieka i wezwij helikopter. Masz 300 punktów akcji — nie przekrocz limitu!"}
]

MAX_STEPS = 50
flag = None

for step in range(MAX_STEPS):
    print(f"\n{'='*60}")
    print(f"Krok {step + 1} | Akcje do tej pory: {len(action_log)}")
    print('='*60)

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=8192,
        system=SYSTEM_PROMPT,
        tools=TOOLS,
        messages=messages,
    )

    messages.append({"role": "assistant", "content": response.content})

    for block in response.content:
        if hasattr(block, 'text') and block.text:
            print(f"\n[Agent]: {block.text[:600]}")

    if response.stop_reason == "end_turn":
        print("\nAgent zakończył pracę.")
        break

    if response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"\n[Narzędzie]: {block.name}")
                result = handle_tool(block.name, block.input)

                if "FLG:" in result or "{FLG" in result:
                    flag = result

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })

        messages.append({"role": "user", "content": tool_results})

        if flag:
            print(f"\n{'*'*60}")
            print(f"FLAGA ZNALEZIONA: {flag}")
            print('*'*60)
            break

print(f"\nMisja zakończona. Łączna liczba akcji: {len(action_log)}")

In [ ]:
# Wyświetl wynik
if flag:
    print(f"Flaga: {flag}")
else:
    for msg in messages:
        content = str(msg.get('content', ''))
        if 'FLG' in content:
            print(f"Flaga w historii: {content[:500]}")
            break